In [ ]:
import pandas as pd
import ast
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# --- 1. Load your actual data ---
# We try to load the file generated by your scraper. 
# If you have the dataframe in memory as 'df', you can skip this line.
try:
    df_raw = pd.read_csv("results/paris_airbnb_final.csv")
    print("Loaded data from 'paris_airbnb_final.csv'")
except FileNotFoundError:
    print("File not found. Please ensure the scraper ran successfully or update the filename.")
    # Fallback: create an empty DF to prevent crash if file is missing
    df_raw = pd.DataFrame()

# --- 2. The Organization Function ---
def organize_airbnb_data(df):
    """
    Takes the raw dataframe, fixes types, and one-hot encodes amenities.
    """
    if df.empty:
        print("Dataframe is empty. Cannot organize.")
        return df

    print(f"--- Organizing {len(df)} listings ---")
    
    # Work on a copy to avoid SettingWithCopy warnings
    df = df.copy()

    # A. Fix the 'amenities_list' column
    # It looks like strings: "['Wifi', 'Kitchen']" -> We need actual lists: ['Wifi', 'Kitchen']
    def safe_eval(x):
        try:
            # If it's already a list, return it
            if isinstance(x, list): return x
            # If it's a string, evaluate it
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []

    if 'amenities_list' in df.columns:
        df['amenities_clean'] = df['amenities_list'].apply(safe_eval)
    else:
        print("Warning: 'amenities_list' column missing.")
        df['amenities_clean'] = []
    
    # B. One-Hot Encoding
    # Turns ['Wifi', 'TV'] into columns: has_wifi=1, has_tv=1
    mlb = MultiLabelBinarizer()
    
    # Fit and transform
    amenities_matrix = mlb.fit_transform(df['amenities_clean'])
    
    # Create clean column names (e.g., "Has Wifi" -> "has_wifi")
    new_columns = [f"has_{c.lower().replace(' ', '_').replace('-', '_')}" for c in mlb.classes_]
    
    amenities_df = pd.DataFrame(
        amenities_matrix, 
        columns=new_columns, 
        index=df.index
    )
    
    # C. Merge back
    final_df = pd.concat([df, amenities_df], axis=1)
    
    # D. Cleanup
    # Convert host_since to datetime
    if 'host_since' in final_df.columns:
        final_df['host_since'] = pd.to_datetime(final_df['host_since'], errors='coerce')
        
    # Drop original messy amenity columns
    cols_to_drop = ['amenities_list', 'amenities_clean']
    final_df = final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns])
    
    print(f"Success! Created {len(new_columns)} binary amenity columns.")
    return final_df

# --- 3. Execution ---
if __name__ == "__main__":
    if not df_raw.empty:
        # Run the organizer
        df_ready = organize_airbnb_data(df_raw)
        
        # Save the machine-learning-ready file
        output_file = "paris_airbnb_ML_ready.csv"
        df_ready.to_csv(output_file, index=False)
        
        print(f"\nSaved processed data to: {output_file}")
        
        # Show a sneak peek of the new columns
        # Filter for columns starting with 'has_'
        amenity_cols = [c for c in df_ready.columns if c.startswith('has_')]
        print(f"Example Amenity Columns: {amenity_cols[:5]}")
        print(df_ready[['name', 'price_night'] + amenity_cols[:3]].head())